# S3Retriever Usage Guide

Two main workflows:

1. **MI values** -- retrieve precomputed mutual information scores for a single experiment or sweep across algorithms / sizes / qualities
2. **Embeddings + signals** -- retrieve matched (embedding, signal) pairs for recomputing MI with a different estimator

In [ ]:
import pandas as pd
import numpy as np

from scaling_laws.s3_retriever import S3Retriever, EmbeddingSignalResult

data = S3Retriever("/home/igor/noise_scaling/data")

## 1. Retrieving MI values

### Single experiment

In [ ]:
mi = data.load_mutual_information(
    "merfish", num_cells=7113, quality=1.0,
    algorithm="Geneformer", signal="ng_idx", seed=42,
)
print(f"MI = {mi:.4f} nats")

### Sweep: compare algorithms for a fixed (dataset, size, quality)

In [ ]:
rows = []
for algo in data.list_algorithms():
    for signal in data.list_signals("merfish"):
        try:
            mi = data.load_mutual_information(
                "merfish", num_cells=7113, quality=1.0,
                algorithm=algo, signal=signal, seed=42,
            )
            rows.append({"algorithm": algo, "signal": signal, "mi": mi})
        except FileNotFoundError:
            pass

pd.DataFrame(rows).pivot(index="algorithm", columns="signal", values="mi")

### Bulk sweep: all sizes and qualities for a dataset

`collect_all_mi_results` returns a DataFrame over all (size, quality, algorithm, signal, seed) combinations.

In [ ]:
mi_df = data.collect_all_mi_results("merfish", algorithms=["PCA", "Geneformer"])
print(f"{len(mi_df)} MI results collected")
mi_df.head(10)

## 2. Retrieving matched embeddings + signal data

For recomputing MI with a different estimator you need both the embedding matrix and the corresponding signal vector/matrix. Rows are aligned: row *i* in the embedding corresponds to row *i* in the signal DataFrame.

### Single experiment

In [ ]:
# Load embedding and its matched signal separately
emb = data.load_embeddings("merfish", num_cells=7113, quality=1.0, algorithm="PCA")
sig = data.load_test_signal("merfish", quality=1.0, signal="ng_idx", algorithm="PCA")

print(f"Embedding shape: {emb.shape}")
print(f"Signal shape:    {sig.shape}")
print(f"Signal columns:  {list(sig.columns)}")
print()
print("Embedding (first 3 rows):")
display(emb.head(3))
print("Signal (first 3 rows):")
display(sig.head(3))

### Sweep with `iter_embeddings_with_signals`

The iterator yields an `EmbeddingSignalResult` dataclass for every (dataset, size, quality, algorithm, signal) combination. Missing files are silently skipped. The tqdm bar shows the current configuration being loaded.

In [ ]:
# Compare all algorithms on merfish at a single (size, quality),
# getting matched embedding + signal for each.
for result in data.iter_embeddings_with_signals(
    datasets=["merfish"],
    sizes={"merfish": [7113]},
    qualities={"merfish": [1.0]},
):
    print(
        f"{result.algorithm:20s} signal={result.signal:<10s}  "
        f"emb={str(result.embedding.shape):<16s}  "
        f"signal_data={str(result.signal_data.shape)}"
    )

### Filtered sweep across sizes

Filter by algorithm and signal to get one (embedding, signal) pair per training size -- useful for plotting MI vs. dataset size with a new estimator.

In [ ]:
for result in data.iter_embeddings_with_signals(
    datasets=["merfish"],
    qualities={"merfish": [1.0]},
    algorithms=["Geneformer"],
    signals={"merfish": ["ng_idx"]},
):
    X = result.embedding.values         # (n_test_cells, embedding_dim)
    Y = result.signal_data              # (n_test_cells, ...)

    # Plug X, Y into your MI estimator here, e.g.:
    #   mi = my_estimator(X, Y)

    print(
        f"n={result.num_cells:<8d}  "
        f"X {str(X.shape):<16s}  "
        f"Y {str(Y.shape)}"
    )